In [1]:
# Perceptron Multicapa

Normlaizamos los datos para el entrenamiento   

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd 

def train_multilayer_perceptron(X, y): 
    alphas = [.001, .01, .1, 1, 10, 100]
    layers = [
        (50, 100, 50, 25, 10), 
        (100, 50, 25, 10), 
        (50, 25, 10),
        (25, 10)
    ]
    activations = ['identity', 'logistic', 'tanh', 'relu']

    rows = []
    for a in alphas: 
        for ly in layers: 
            for activation in activations: 
                
                accs_train, accs_test, f1s, recalls = [], [], [], []
                
                kf = KFold(n_splits=5)
                for train_index, test_index in kf.split(X): 
                    xtrain, xval = X[train_index], X[test_index]
                    ytrain, yval = y[train_index], y[test_index]
                    
                    scaler = StandardScaler()
                    xtrain, xval = scaler.fit_transform(xtrain), scaler.transform(xval)
                
                    model = MLPClassifier(
                        activation=activation, 
                        hidden_layer_sizes=ly,
                        alpha=a, 
                        random_state=0
                    )
                    
                    model.fit(xtrain, ytrain)
                    
                    prediction_train = model.predict(xtrain)
                    prediction_val = model.predict(xval)
                    
                    accs_train.append(accuracy_score(ytrain, prediction_train))
                    accs_test.append(accuracy_score(yval, prediction_val))
                    f1s.append(f1_score(yval, prediction_val))
                    recalls.append(recall_score(yval, prediction_val))
                
                rows.append({
                    'capas ocultas': str(ly),
                    'funcion de activacion': activation,
                    'alpha de regularizacion': a,
                    'media acc train': np.mean(accs_train),
                    'media acc test': np.mean(accs_test), 
                    'IC acc test': np.std(accs_test), 
                    'media f1': np.mean(f1s),
                    'IC f1': np.std(f1s),
                    'media recall': np.mean(recalls), 
                    'IC recall': np.std(recalls)
                })
    results = pd.DataFrame(rows)
    return results.sort_values(by="media f1", ascending=False)

Para datos no filtrados

In [ ]:
from libs.data_configuration import get_data_partitioned

X_train, X_test, y_train, y_test = get_data_partitioned() # 20% para validacion por defecto 

train_multilayer_perceptron(X_train, y_train)

/home/brayan/dev/ml/ModelosProyectoAula/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/brayan/dev/ml/ModelosProyectoAula/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/brayan/dev/ml/ModelosProyectoAula/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/brayan/dev/ml/ModelosProyectoAula/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and 

AttributeError: 'NoneType' object has no attribute 'head'

Para los datos filtrados

In [ ]:
from libs.data_configuration import get_filtered_data_partitioned

X_train, X_test, y_train, y_test = get_filtered_data_partitioned() # 20% para validacion por defecto 

train_multilayer_perceptron(X_train, y_train).head(10)